In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import mlflow
from mlflow.models.signature import infer_signature
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from src.dataset.milho_dataset import get_dataloaders_milho
from src.evaluate import evaluate_model

In [0]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"

best_run_id = "39efe20dccaf4baf9c33e4e128c93067"

In [0]:
_, _, test_loader, classes = get_dataloaders_milho(data_dir=data_dir)

numero_classes = len(classes)

print("Classes:", classes)

In [0]:
model_uri = f"runs:/{best_run_id}/model"
model = mlflow.pytorch.load_model(model_uri)

model.to(device)
model.eval()

print("Modelo carregado com sucesso!")

In [0]:
result = evaluate_model(
    model,
    test_loader,
    device,
    class_names=classes
)

print("Test Accuracy:", result["accuracy"])
print(result["classification_report"])

In [0]:
cm = result["confusion_matrix"]

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - Teste")

plt.savefig("figures/confusion_matrix_test.png")
plt.show()

In [0]:
mlflow.set_experiment("/Shared/Milho_Doencas")

with mlflow.start_run(run_name="evaluation_best_model"):

    mlflow.log_param("evaluated_run_id", best_run_id)

    mlflow.log_metric("test_accuracy", result["accuracy"])

    mlflow.log_artifact("figures/confusion_matrix_test.png")

    with open("classification_report.txt", "w") as f:
        f.write(result["classification_report"])

    mlflow.log_artifact("classification_report.txt")

    print("Avaliação logada no MLflow!")

In [0]:
pyfunc_run_id = "7e4caf958d4a4b5d8a3a7177989914c2"
test_model_uri = f"runs:/{pyfunc_run_id}/model"

loaded_pyfunc_model = mlflow.pyfunc.load_model(test_model_uri)

data_dir = "/Volumes/workspace/agro-diasease/data/dados_divididos"
_, _, test_loader, classes = get_dataloaders_milho(data_dir=data_dir)

image, label = test_loader.dataset[1]

prediction = loaded_pyfunc_model.predict(
    image.unsqueeze(0).numpy()
)
label_real = classes[label]

print("\n✅ Teste de carregamento bem-sucedido!")
print(f"Predição: {prediction[0]['prediction']}")
print(f"Confiança: {prediction[0]['confidence']:.4f}")
print(f"Label real: {label_real}")

In [0]:
# Registrar o modelo PyFunc no Unity Catalog
model_name = "modelo_milho_doencas_pyfunc"
model_uri = f"runs:/{pyfunc_run_id}/model"

print(f"Registrando modelo no Unity Catalog: {model_name}")

# Registrar nova versão
model_version = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"\n✅ Modelo registrado como versão {model_version.version}")

# Definir alias @production
client = mlflow.tracking.MlflowClient()
client.set_registered_model_alias(
    name=model_name,
    alias="production",
    version=model_version.version
)

print(f"✅ Alias '@production' definido para versão {model_version.version}")
print(f"\n📌 Para usar na API:")
print(f"   MODEL_URI = 'models:/{model_name}@production'")
print(f"   model = mlflow.pyfunc.load_model(MODEL_URI)")

In [0]:
import mlflow

model_uri = "models:/workspace.default.modelo_milho_doencas_pyfunc@production"

local_path = mlflow.artifacts.download_artifacts(model_uri)

print(local_path)

In [0]:
display(dbutils.fs.ls("/local_disk0/user_tmp_data/spark-a0c46369-df62-43b0-85dc-37/tmpiwv797yu/"))

In [0]:
model_uri = f"runs:/{best_run_id}/model"

mlflow.register_model(
    model_uri,
    "modelo_milho_doencas"
)

print("Modelo registrado com sucesso!")

In [0]:
client = mlflow.tracking.MlflowClient()

client.set_registered_model_alias(
    name="modelo_milho_doencas",
    alias="production",
    version=1
)